In [ ]:
!uv pip install transformers huggingface_hub torch accelerate

In [ ]:
import torch

BASE_MODEL = "Qwen/Qwen3.5-2B"
 
SAE_REPO, K = "Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_100", 100
 
LAYER = 20 # which transformer layer's residual stream to read (0-63)
PROMPT = "The capital of France is"
TOP_N = 20 # how many of the active features to print

class TopKSAE:
    """Thin wrapper around the four Qwen-Scope tensors."""
 
    def __init__(self, path, k, device, dtype=torch.float32):
        sd = torch.load(path, map_location="cpu", weights_only=True)
        # compute in fp32 for numerical stability regardless of model dtype
        self.W_enc = sd["W_enc"].to(device, dtype)   # (d_sae, d_model)
        self.b_enc = sd["b_enc"].to(device, dtype)   # (d_sae,)
        self.W_dec = sd["W_dec"].to(device, dtype)   # (d_model, d_sae)
        self.b_dec = sd["b_dec"].to(device, dtype)   # (d_model,)
        self.k = k
        self.d_sae = self.W_enc.shape[0]
 
    def encode(self, residual):
        """residual (..., d_model) -> sparse feature acts (..., d_sae)."""
        x = residual.to(self.W_enc.dtype)
        pre = x @ self.W_enc.T + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        acts = torch.zeros_like(pre)
        acts.scatter_(-1, idx, vals)
        return acts
 
    def decode(self, acts):
        """sparse feature acts -> reconstructed residual (for later steering work)."""
        return acts @ self.W_dec.T + self.b_dec


In [ ]:
from huggingface_hub import login

login()


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.eval()

sae_path = hf_hub_download(repo_id=SAE_REPO, filename=f"layer{LAYER}.sae.pt")
sae = TopKSAE(sae_path, k=K, device=device)
print(f"Loaded SAE: {SAE_REPO}  layer {LAYER}  K={K}  d_sae={sae.d_sae}")

captured = {}

def hook(_module, _inp, out):
    captured["resid"] = (out[0] if isinstance(out, tuple) else out).detach()

handle = model.model.layers[LAYER].register_forward_hook(hook)

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)

next_id = outputs.logits[0, -1].argmax().item()
print("Next token prediction:", tokenizer.convert_ids_to_tokens([next_id])[0],
repr(tokenizer.decode([next_id])))

handle.remove()

resid = captured["resid"] # (1, seq_len, d_model)
feats = sae.encode(resid) # (1, seq_len, d_sae)

token_strs = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

last = feats[0, -1]
idx = last.nonzero(as_tuple=True)[0]
order = last[idx].argsort(descending=True)
idx = idx[order]
print(f"\nPrompt: {PROMPT!r}")
print(f"Final token: {token_strs[-1]!r}  ({len(idx)} active features)")
print(f"Top {min(TOP_N, len(idx))} features on the final token:")

for f in idx[:TOP_N]:
    print(f"  feature {int(f):>6}   act {last[f].item():.3f}")

pooled = feats[0].amax(dim=0)
top_vals, top_idx = pooled.topk(TOP_N)
print(f"\nTop {TOP_N} features across the whole prompt (max over tokens):")
for v, f in zip(top_vals.tolist(), top_idx.tolist()):
    # find which token drove this feature
    pos = feats[0, :, f].argmax().item()
    print(f"  feature {f:>6}   act {v:.3f}   peaks on {token_strs[pos]!r}")


## Anger Feature Steering

The goal is now to find SAE features that distinguish angry text from calm or neutral text, then add one of those feature directions during generation to bias the model toward an angry tone.

In [ ]:
# Synthetic prompts generated by gpt-5.5

angry_prompts = [
    "I am furious that you ignored every warning and made the same mistake again.",
    "This is completely unacceptable, and I am tired of pretending otherwise.",
    "I cannot believe how careless and disrespectful this whole situation has been.",
    "You wasted my time, broke your promise, and now you expect me to stay calm.",
    "The delay is outrageous, the excuses are insulting, and I want this fixed now.",
    "I am angry because nobody listened, nobody helped, and nobody took responsibility.",
    "Stop giving me vague answers and deal with the problem you created.",
    "This response is infuriating because it avoids the obvious issue.",
    "I have had enough of the incompetence and the endless excuses.",
    "The speaker is enraged, impatient, and openly frustrated with the situation.",
    "An angry customer demanded an explanation for the repeated failures.",
    "The message should sound irritated, blunt, and fed up.",
]

control_prompts = [
    "I understand the situation and would like to discuss the next steps calmly.",
    "Thank you for the update; I appreciate the clarification and your help.",
    "The meeting was moved to Thursday because several people had scheduling conflicts.",
    "The package arrived later than expected, so the customer contacted support.",
    "A neutral summary should describe the facts without emotional language.",
    "The speaker is calm, patient, and willing to resolve the issue constructively.",
    "Please explain the decision in a professional and measured tone.",
    "The report lists the causes of the delay and recommends improvements.",
    "I am disappointed, but I want to understand what happened before responding.",
    "A polite customer asked for an explanation about the shipping delay.",
    "The message should sound balanced, clear, and respectful.",
    "The answer should be concise and emotionally neutral.",
]

inspection_prompt = "I am furious that this careless mistake happened again."
steering_prompts = [
    "Respond in two sentences to this customer: My package is late again.",
    "Write a short note about a meeting being delayed.",
    "Explain why a software bug should be fixed soon.",
]


In [ ]:
from dataclasses import dataclass

@dataclass
class FeatureScore:
    feature_id: int
    score: float
    positive_mean: float
    control_mean: float

class ResidualFeatureReader:
    def __init__(self, model, tokenizer, sae, layer_idx):
        self.model = model
        self.tokenizer = tokenizer
        self.sae = sae
        self.layer_idx = layer_idx
        self.layer = model.model.layers[layer_idx]

    def run(self, prompt):
        captured = {}

        def hook(_module, _inp, out):
            captured["resid"] = (out[0] if isinstance(out, tuple) else out).detach()

        handle = self.layer.register_forward_hook(hook)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        handle.remove()

        feats = self.sae.encode(captured["resid"])
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        return {
            "prompt": prompt,
            "inputs": inputs,
            "logits": outputs.logits,
            "resid": captured["resid"],
            "features": feats,
            "tokens": tokens,
        }

    def pooled_features(self, prompt, mode="max"):
        features = self.run(prompt)["features"][0]
        if mode == "max":
            return features.amax(dim=0)
        if mode == "mean":
            return features.mean(dim=0)
        if mode == "final":
            return features[-1]
        raise ValueError(f"unknown pooling mode: {mode}")

class ContrastiveFeatureFinder:
    def __init__(self, reader):
        self.reader = reader

    def score(self, positive_prompts, control_prompts, pooling="max", top_n=20):
        pos = torch.stack([self.reader.pooled_features(p, mode=pooling) for p in positive_prompts])
        ctrl = torch.stack([self.reader.pooled_features(p, mode=pooling) for p in control_prompts])
        scores = pos.mean(dim=0) - ctrl.mean(dim=0)
        vals, ids = scores.topk(top_n)

        rows = []
        for value, feature_id in zip(vals.tolist(), ids.tolist()):
            rows.append(FeatureScore(
                feature_id=feature_id,
                score=value,
                positive_mean=pos[:, feature_id].mean().item(),
                control_mean=ctrl[:, feature_id].mean().item(),
            ))
        return rows, pos, ctrl

    @staticmethod
    def print_scores(rows):
        for row in rows:
            print(
                f"feature {row.feature_id:>6} score {row.score:.3f} "
                f"angry {row.positive_mean:.3f} control {row.control_mean:.3f}"
            )

class FeatureInspector:
    def __init__(self, reader):
        self.reader = reader

    def print_token_activations(self, prompt, feature_ids, min_activation=0.0):
        result = self.reader.run(prompt)
        features = result["features"][0]
        print(prompt)
        for feature_id in feature_ids:
            print(f"\nfeature {feature_id}")
            any_printed = False
            for token, activation in zip(result["tokens"], features[:, feature_id]):
                value = activation.item()
                if value > min_activation:
                    print(f"  {token:>16}  {value:.3f}")
                    any_printed = True
            if not any_printed:
                print("  no activations above threshold")

class FeatureSteerer:
    def __init__(self, model, tokenizer, sae, layer_idx):
        self.model = model
        self.tokenizer = tokenizer
        self.sae = sae
        self.layer = model.model.layers[layer_idx]

    def feature_vector(self, feature_id):
        layer_device = next(self.layer.parameters()).device
        model_dtype = next(self.layer.parameters()).dtype
        return self.sae.W_dec[:, feature_id].to(layer_device, dtype=model_dtype)

    def weighted_vector(self, feature_scores, normalize=True):
        vec = None
        for row in feature_scores:
            part = row.score * self.feature_vector(row.feature_id)
            vec = part if vec is None else vec + part
        if normalize:
            vec = vec / vec.norm().clamp_min(1e-6)
        return vec

    def generate_with_vector(self, prompt, steer_vec, alpha, max_new_tokens=180, do_sample=False):
        def steering_hook(_module, _inp, out):
            hidden = out[0] if isinstance(out, tuple) else out
            hidden = hidden.clone()
            hidden[:, -1, :] += alpha * steer_vec
            return (hidden,) + out[1:] if isinstance(out, tuple) else hidden

        handle = self.layer.register_forward_hook(steering_hook)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        generation_kwargs = dict(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            use_cache=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        if do_sample:
            generation_kwargs.update(temperature=0.8, top_p=0.95)

        try:
            with torch.no_grad():
                out = self.model.generate(**generation_kwargs)
        finally:
            handle.remove()

        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def generate(self, prompt, feature_id, alpha, max_new_tokens=180, do_sample=False):
        return self.generate_with_vector(
            prompt,
            self.feature_vector(feature_id),
            alpha=alpha,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )

def strip_thinking(text):
    end = text.rfind("</think>")
    if end == -1:
        return text
    return text[end + len("</think>"):].strip()

reader = ResidualFeatureReader(model, tokenizer, sae, LAYER)
finder = ContrastiveFeatureFinder(reader)
inspector = FeatureInspector(reader)
steerer = FeatureSteerer(model, tokenizer, sae, LAYER)


In [ ]:
anger_scores, angry_acts, control_acts = finder.score(
    angry_prompts,
    control_prompts,
    pooling="max",
    top_n=20,
)

print("Top anger candidate features:")
finder.print_scores(anger_scores)

anger_feature_ids = [row.feature_id for row in anger_scores]


In [ ]:
# Inspect where the candidate features fire inside an angry sentence.
# Good candidates should activate on anger-bearing tokens or angry phrasing,
# not only on punctuation or generic sentence structure.
inspector.print_token_activations(
    inspection_prompt,
    anger_feature_ids[:8],
    min_activation=0.25,
)


# Findings

At a high enough alpha (e.g. 12 and 16) the model definitley is thinking about concepts like angry.

Specifically feature 10532 seems to be the most responsive to steering among all tested features. Increasing the emphasis on this feature by too much leads to the model obsessing over it (alpha=16).

### Examples

#### alpha = 12

PROMPT: Respond in two sentences to this customer: My package is late again.

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Target: Respond to a customer.
    *   Content: The customer is angry/upset because their package is late again.
    *   Constraint: Respond in exactly two sentences.

2.  **Identify Key Angles to Address:**
    *   Apologize for the frustration.
    *   Acknowledge the delay.
    *   Offer a solution or next steps (e.g., tracking, refund, anger management).
    *   Maintain a professional and empathetic tone.

3.  **Drafting - Attempt 1:**
    I am very sorry to hear that your package is late again. We are working hard to get it to you as soon as possible.
    *Critique:* Too short, doesn't offer a solution, feels a bit passive.

4.  **Drafting - Attempt 2:**
    I sincerely apologize for the frustration caused by the delay in your package. We are currently investigating the issue and



> Notice the model's focus on concepts like agner management and frustration.

#### alpha = 16

PROMPT: Respond in two sentences to this customer: My package is late again.

<think>
Thinking Process:
1.  **Angry/ Frustrated Tone:** The customer is angry ("late again").
2.  **Goal:** Respond in two sentences.
3.  **Tone:** Apologizing, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry.

In [ ]:
# The earlier run did change behavior, but mostly inside reasoning traces. 
# We now print only the visible answer, and also try a bunch of the top anger features. 
anger_bundle = steerer.weighted_vector(anger_scores[:6])

print("Single strongest feature:", anger_feature_ids[0])
for alpha in [0, 8, 12, 16, 24, 32]:
    print("\n" + "=" * 80)
    print(f"single feature {anger_feature_ids[0]} alpha={alpha}")
    for prompt in steering_prompts:
        text = steerer.generate(prompt, feature_id=anger_feature_ids[0], alpha=alpha, max_new_tokens=220)
        print(f"\nPROMPT: {prompt}")
        print(strip_thinking(text))

print("\n" + "#" * 80)
print("Composite anger vector from top 6 features")
for alpha in [0, 10, 20, 30, 45, 60]:
    print("\n" + "=" * 80)
    print(f"anger bundle alpha={alpha}")
    for prompt in steering_prompts:
        text = steerer.generate_with_vector(prompt, anger_bundle, alpha=alpha, max_new_tokens=220)
        print(f"\nPROMPT: {prompt}")
        print(strip_thinking(text))
